# Benchmark validation report

Can the metrics detect known defects, and what do the underlying tasks actually cover? This report regenerates correctness results for all seven local synthetic domains and the retail, airline, and telecom τ²-bench base splits. It also runs constructed clean controls, the original harness demo, a recorded-run sample when available, and tool-diversity perturbations on both datasets.

The attached *Metric specification, Milestone 1 + Milestone 2* motivates two experiments: inject mistakes with known effects on correctness, and concentrate tool calls to test whether diversity drops. Its November 4 progress report covered three τ² domains, three control sets, and one synthetic domain. The static AST audit evolved into the synthetic evaluator, and the original prototype evolved into the shared corruption harness. The results below come from the current inputs and code. Section 7 evaluates semantic correctness with a single LLM judge and shows its behavior on independently labeled controls.

### Reproduce

Run `uv sync --locked --extra notebook --extra tau2` from the repository root, then open this notebook with the project kernel. A sibling `tau2-bench` checkout at revision `5ade54e282a4555e4d6c52d6b9b861ea8872afa6` is detected automatically. For another checkout, set `TAU2_BENCHMARK_ROOT` or edit the setup cell. Run all cells. The default run replays reference or already recorded actions offline and reads the saved judge report when available. Section 7 can regenerate judge results on demand.

In [ ]:
import csv
import io
import json
import os
import subprocess
import sys
from collections import Counter
from math import isclose
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import pandas as pd
import seaborn as sns
from IPython.display import HTML, Markdown, display

cwd = Path.cwd()
ROOT = next((p for p in (cwd, *cwd.parents)
             if (p / "validation" / "correctness_checker.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook from inside the evaluation repository.")
sys.path.insert(0, str(ROOT))

from correctness_op.deterministic_audit import SYSTEM
from evaluation.tau2.runtime import context
from evaluation.tool_usage_diversity import calculate_diversity, validate_perturbations
from validation.reporting import findings

TAU_ROOT = Path(os.environ.get("TAU2_BENCHMARK_ROOT", ROOT.parent / "tau2-bench")).expanduser().resolve()
if not TAU_ROOT.is_dir():
    raise FileNotFoundError(f"Set TAU2_BENCHMARK_ROOT to the pinned checkout. Tried {TAU_ROOT}")
SEED = 42
RATES = (0, .1, .25, .5, 1)
DOMAINS = ("retail", "airline", "telecom")
SYNTHETIC = sorted(path.parent for path in (ROOT / "data").glob("*/test_cases.json"))
# Optional reviewed original-task labels, keyed by τ² domain.
LABELS = {}
# Use a saved retail recording from the pinned checkout when available.
RECORDING = TAU_ROOT / "data/tau2/results/final/gpt-4.1-2025-04-14_retail_default_gpt-4.1-2025-04-14_4trials.json"
RECORDED_DOMAIN = "retail"
RECORDED_LIMIT = 5
# The bundled historical sample embeds older task definitions. Use False for a recording
# from the current task definitions, or True to intentionally preserve historical tasks.
RECORDED_ALLOW_TASK_DRIFT = True
REPEAT_ALL = False  # The demo always runs twice. Enable to repeat every CLI experiment.
RUN_TESTS = True

output_root = ROOT / "output" / "validation-notebook"
output_root.mkdir(parents=True, exist_ok=True)
RUN_DIR = Path(mkdtemp(prefix="run-", dir=output_root))
sns.set_theme(style="whitegrid", palette="colorblind")
print(f"τ² source: {TAU_ROOT}")
print(f"Artifacts: {RUN_DIR}")

τ² source: /Users/andersonchan/Documents/Code Projects/BTT/tau2-bench
Artifacts: /Users/andersonchan/Documents/Code Projects/BTT/Automation-Anywhere-2B-evaluation-framework-for-synthetic-agent-benchmark-datasets/output/validation-notebook/run-a50rlggm


## 1. How to read correctness

| Check | Synthetic fixture evidence | τ²-bench evidence |
| --- | --- | --- |
| Schema | Task fields, types, milestones | Upstream task and action structure |
| Signatures | YAML tool declarations and Python mock signatures | Real tool schemas, by requestor |
| References | JSON/Python pairing and identifier provenance | Entity existence and ownership in the evolving database |
| Output consistency | Statically inferable mock output and fixture expectations | Real replay, preconditions, and recorded-output comparison when supplied |

A task score is the fraction of its **available checks** that pass. For example, `[1, 1, 0, null]` scores **2/3** with **3/4 check coverage**. The dataset score averages task scores. The task pass rate counts tasks with a score of 1. The evaluator separately reports full coverage, so a 100% task score can still reflect only part of the evidence.

The synthetic reader parses fixture code without importing the agent. τ² replays real tools from a fresh environment. A synthetic static score and a τ² replay score answer different questions. Independent executable outcome assertions are reported separately from correctness.

In [ ]:
runs = {}
artifact_names = ("report.json", "manifest.json", "summary.csv", "report.md")
validation_args = ["--seed", str(SEED), "--rates", *map(str, RATES)]


def run_cli(name, module, args, parent=RUN_DIR):
    directory = parent / name
    command = [sys.executable, "-m", module, *map(str, args), "--output", str(directory)]
    result = subprocess.run(command, cwd=ROOT, capture_output=True, text=True)
    if result.returncode not in (0, 1, 3) or not (directory / "report.json").is_file():
        raise RuntimeError(f"{name} exited {result.returncode}\n{result.stdout}\n{result.stderr}")
    report = json.loads((directory / "report.json").read_text())
    expected_code = {"pass": 0, "not_run": 0, "fail": 1, "inconclusive": 3}[report["status"]]
    if expected_code != result.returncode:
        raise ValueError(f"{name}: report status disagrees with exit code")
    return report, directory


def collect(name, family, module, args, repeat=False):
    report, directory = run_cli(name, module, args)
    if repeat or REPEAT_ALL:
        _, repeated = run_cli(name, module, args, RUN_DIR / "repeat")
        assert all((directory / file).read_bytes() == (repeated / file).read_bytes()
                   for file in artifact_names), f"{name}: repeated artifacts differ"
    runs[name] = (family, report, directory)
    print(f"{name}: {report['status']} ({report['task_count']} tasks)", flush=True)


collect("Harness demo", "Demo", "validation.correctness_checker", ["--demo", *validation_args], repeat=True)
assert runs["Harness demo"][1]["status"] == "pass", "The built-in harness demo should pass"

for dataset in SYNTHETIC:
    collect(dataset.name, "Synthetic static", "evaluation.synthetic",
            ["--dataset", dataset, "--validate-corruptions", *validation_args])

for domain in DOMAINS:
    args = ["--benchmark-root", TAU_ROOT, "--domain", domain, "--validate-corruptions", *validation_args]
    collect(f"τ² {domain}", "τ² original", "evaluation.tau2",
            [*args, "--labels", Path(LABELS[domain]).expanduser().resolve()] if domain in LABELS else args)
    collect(f"τ² {domain} controls", "τ² controls", "evaluation.tau2", [*args, "--controls"])

if RECORDING.is_file():
    collect(f"τ² {RECORDED_DOMAIN} recorded", "τ² recorded", "evaluation.tau2.recorded",
            ["--benchmark-root", TAU_ROOT, "--domain", RECORDED_DOMAIN,
             "--results", RECORDING.resolve(), "--limit", RECORDED_LIMIT,
             *(["--allow-task-drift"] if RECORDED_ALLOW_TASK_DRIFT else [])])
else:
    print("Recorded replay unavailable. Set RECORDING to a compatible half-duplex results.json.")

Harness demo: pass (20 tasks)
banking_accounts_deposits: pass (8 tasks)
banking_customer_onboarding: pass (6 tasks)
banking_fraud_monitoring: pass (13 tasks)
healthcare_patient_onboarding: pass (9 tasks)
healthcare_prior_authorization: pass (9 tasks)


## 2. Verify the experiments

For each campaign, the harness corrupts a seeded, nested subset of eligible tasks. Eligibility requires a constructible mutation and a passing target check at baseline. For example, with 97 eligible tasks out of 114, a requested 25% corrupts 24 tasks, or 21.1% of the whole dataset. The expected score follows the declared check changes, including checks that become unavailable after an earlier failure.

The cell independently rebuilds scores, detection counts, and clean false positives from the saved per-task audit and manifest. It checks nesting and compares JSON row counts with CSV. The demo also compares all four files across two fresh runs. A **FAIL** or **INCONCLUSIVE** result remains visible even when the report files are internally consistent.

In [ ]:
from collections import defaultdict
from statistics import mean


def close(actual, expected):
    if actual is None or expected is None:
        return actual is expected
    return isclose(actual, expected, rel_tol=0, abs_tol=1e-12)


def score(vector):
    return mean(value for value in vector.values() if value is not None)


manifests = {}
for name, (_, report, directory) in runs.items():
    manifest = manifests[name] = json.loads((directory / "manifest.json").read_text())
    baseline = report["baseline_results"]
    assert len(baseline) == len(manifest["tasks"]) == report["task_count"], name
    assert close(report["baseline_score"], mean(score(r["checks"]) for r in baseline.values())), name
    labeled = {t["id"] for t in manifest["tasks"] if t.get("expected_pass") is True}
    baseline_fp = sum(score(baseline[key]["checks"]) < 1 for key in labeled)
    assert close(report["baseline_false_positive_rate"], baseline_fp / len(labeled) if labeled else None), name
    csv_rows = list(csv.DictReader(io.StringIO((directory / "summary.csv").read_text())))
    assert len(csv_rows) == (len(report["campaigns"]) or len(baseline)), name

    audit = defaultdict(list)
    for item in report["audit"]:
        audit[item["kind"], item["rate"]].append(item)
    previous = defaultdict(set)
    for row in sorted(report["campaigns"], key=lambda row: (row["kind"], row["requested_rate"])):
        kind, rate = row["kind"], row["requested_rate"]
        context_label = (name, kind, rate)
        items = audit[kind, rate]
        selected = {item["task_id"] for item in items if item["corrupted"]}
        assert len(items) == len(baseline) and {item["task_id"] for item in items} == baseline.keys(), context_label
        assert previous[kind] <= selected, context_label
        previous[kind] = selected
        assert len(selected) == row["corrupted_tasks"] == int(row["eligible_tasks"] * rate + .5), context_label
        assert close(row["effective_dataset_rate"], len(selected) / len(baseline)), context_label

        expected, observed = [], []
        detected = targeted = matched = clean_failures = 0
        for item in items:
            original = baseline[item["task_id"]]["checks"]
            result = item["result"]["checks"]
            mutation = item["mutation"]
            predicted = original
            if mutation:
                target = mutation["expected_check"]
                predicted = {**original, **mutation.get("expected_checks", {target: 0})}
                detected += score(result) < score(original)
                targeted += result[target] is not None and result[target] < original[target]
                matched += result == predicted
            elif item["task_id"] in labeled:
                clean_failures += score(result) < 1
            expected.append(score(predicted))
            observed.append(score(result))
        clean = len(labeled - selected)
        assert close(row["expected_score"], mean(expected)) and close(row["observed_score"], mean(observed)), context_label
        assert (row["detected"], row["targeted_detected"], row["expected_check_vectors_matched"]) == (detected, targeted, matched), context_label
        assert row["clean_tasks"] == clean and row["false_positives"] == clean_failures, context_label
        assert close(row["false_positive_rate"], clean_failures / clean if clean else None), context_label

print(f"Reconciled per-task evidence, report summaries, and CSV row counts for {len(runs)} runs.")
print("The harness demo produced byte-identical JSON, CSV, and Markdown artifacts on repeat.")

In [ ]:
if RUN_TESTS:
    result = subprocess.run(
        [sys.executable, "-m", "unittest", "discover", "-s", "validation", "-p", "test_*.py"],
        cwd=ROOT, env={**os.environ, "TAU2_BENCHMARK_ROOT": str(TAU_ROOT)},
        capture_output=True, text=True,
    )
    test_log = result.stdout + result.stderr
    (RUN_DIR / "tests.txt").write_text(test_log)
    print("\n".join(test_log.strip().splitlines()[-6:]))
    if result.returncode:
        raise RuntimeError(f"Validation test suite failed. See {RUN_DIR / 'tests.txt'}")

## 3. Baseline score, coverage, and outcomes

The table keeps the average score, the fraction of tasks that pass applicable checks, and full coverage side by side. The heatmaps show where each check was applicable and where it failed. Counts include all tasks in each row. A gray cell in the failure heatmap means the check had no evaluated tasks. Blocked checks are included in the unavailable counts.

Independent outcome assertions provide a second kind of evidence. For recorded runs, this can include database state, supported communication checks, and normal termination. A recording is a small agent-run sample on the supplied database snapshot. The bundled historical sample retains its embedded task definitions and lists any IDs that differ from the supplied benchmark. Historical database identity remains unverified.

In [ ]:
def pct(value):
    return "—" if pd.isna(value) else f"{value:.1%}"


def title(value):
    return value.replace("_", " ")


overview_rows, check_rows, outcome_rows, task_rows = [], [], [], []
for name, (family, report, _) in runs.items():
    summary, count = report["baseline_summary"], report["task_count"]
    overview_rows.append({
        "Run": name, "Evidence": family, "Tasks": count, "Mean score": report["baseline_score"],
        "Pass applicable": summary["task_pass_rate"], "Full coverage": summary["fully_evaluated_tasks"] / count,
        "Validation": title(report["status"]).upper(),
        "Clean labels": report["clean_labeled_tasks"], "Clean false positives": report["baseline_false_positive_rate"],
    })
    for check, stats in summary["checks"].items():
        check_rows.append({"Run": name, "Check": title(check), **stats,
                           "Coverage": stats["evaluated"] / count,
                           "Failure rate": stats["failed"] / stats["evaluated"] if stats["evaluated"] else None})
    outcome_rows.append({"Run": name, **summary["outcomes"]})
    for task_id, verdict in report["baseline_results"].items():
        task_rows.append({"Run": name, "Task": task_id, "Score": verdict["score"],
                          "Checks available": sum(v is not None for v in verdict["checks"].values()),
                          **verdict["checks"]})

overview = pd.DataFrame(overview_rows).set_index("Run")
checks = pd.DataFrame(check_rows)
tasks = pd.DataFrame(task_rows)
outcomes = pd.DataFrame(outcome_rows).set_index("Run")
display(overview.style.format({col: pct for col in (
    "Mean score", "Pass applicable", "Full coverage", "Clean false positives"
)}))
overview.to_csv(RUN_DIR / "baseline-overview.csv")
tasks.to_csv(RUN_DIR / "baseline-tasks.csv", index=False)
checks.to_csv(RUN_DIR / "check-coverage.csv", index=False)
outcomes.to_csv(RUN_DIR / "outcomes.csv")

rates_for_plot = overview[["Mean score", "Pass applicable", "Full coverage"]]
ax = rates_for_plot.plot.barh(figsize=(11, 7), width=.8)
ax.invert_yaxis()
ax.set(xlim=(0, 1.02), xlabel="Score or share of tasks", ylabel="", title="Baseline correctness and full coverage")
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 6.8), layout="constrained")
for ax, metric, cmap in zip(axes, ("Coverage", "Failure rate"), ("Blues", "Oranges")):
    matrix = checks.pivot(index="Run", columns="Check", values=metric).reindex(overview.index)
    sns.heatmap(matrix, vmin=0, vmax=1, cmap=cmap, annot=True, fmt=".0%",
                cbar_kws={"format": PercentFormatter(1)}, ax=ax, yticklabels=True)
    ax.set(title=metric, xlabel="", ylabel="", facecolor="#eeeeee")
    ax.tick_params(axis="x", rotation=25)
plt.show()

ax = outcomes.rename(columns={"not_evaluated": "not evaluated"}).plot.barh(
    stacked=True, color=["#00796b", "#c4512c", "#b9c1c9"], figsize=(10, 6.5),
)
ax.set(xlabel="Tasks", title="Independent outcome evidence", ylabel="")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 4. How the evaluator responds to known mistakes

Each chart uses requested corruption rate on the horizontal axis. The dashed line shows the expected average score and the dots show the observed score. Matching curves are one part of validation. The detection table also checks whether the intended category failed and whether the full four-check vector matched. A zero-corruption row has no detection denominator.

For labeled clean tasks, false positives use only the uncorrupted clean tasks at that rate. The demo and constructed τ² controls have independent labels. Original τ² tasks only contribute when you supply reviewed labels. An unexercised campaign makes the experiment inconclusive unless a failure already makes it fail.

In [ ]:
campaign_rows = [
    {"Run": name, "Evidence": family, **row}
    for name, (family, report, _) in runs.items() for row in report["campaigns"]
]
campaigns = pd.DataFrame(campaign_rows)
campaigns["vector_match_rate"] = campaigns["expected_check_vectors_matched"].div(
    campaigns["corrupted_tasks"].where(campaigns["corrupted_tasks"] > 0)
)
campaigns["score_error"] = campaigns["observed_score"] - campaigns["expected_score"]
campaigns.to_csv(RUN_DIR / "campaigns.csv", index=False)

for family, table in campaigns.groupby("Evidence", sort=False):
    names = list(table["Run"].unique())
    kinds = sorted(table["kind"].unique())
    fig, axes = plt.subplots(len(names), len(kinds), squeeze=False, sharey=True, sharex=True,
                            figsize=(max(9, 2.35 * len(kinds)), 1.55 * len(names) + 1.6),
                            layout="constrained")
    for index, name in enumerate(names):
        for column, kind in enumerate(kinds):
            ax = axes[index, column]
            subset = table.loc[(table["Run"] == name) & (table["kind"] == kind)].sort_values("requested_rate")
            if subset.empty:
                ax.text(.5, .5, "No campaign", ha="center", va="center", transform=ax.transAxes)
            else:
                ax.plot(subset["requested_rate"], subset["expected_score"],
                        "--", color="#555555", label="Expected")
                sns.scatterplot(data=subset, x="requested_rate", y="observed_score",
                                ax=ax, color="#0072b2", s=31, label="Observed", legend=False)
                if subset["corrupted_tasks"].max() == 0:
                    ax.text(.5, .12, "No eligible tasks", ha="center", transform=ax.transAxes, fontsize=8)
                ax.set(xlabel="", ylabel="")
            if index == 0:
                ax.set_title(title(kind).replace(" ", "\n", 1), fontsize=10)
            if column == 0:
                ax.set_ylabel(name.replace("_", " "), fontsize=8, rotation=0, ha="right", va="center")
            ax.set(ylim=(-.04, 1.07), xticks=RATES if len(RATES) < 5 else (0, .25, .5, 1))
            ax.xaxis.set_major_formatter(PercentFormatter(1, decimals=0))
            ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
            ax.tick_params(labelsize=8)
    from matplotlib.lines import Line2D
    fig.legend(handles=[Line2D([], [], linestyle="--", color="#555555", label="Expected"),
                        Line2D([], [], linestyle="", marker="o", color="#0072b2", label="Observed")],
               loc="outside upper right")
    fig.suptitle(f"{family} · correctness scores", x=.02, ha="left")
    fig.supxlabel("Requested corruption rate")
    plt.show()

# The lowest measured rate across nonempty corruption batches exposes any missed detections.
for family, table in campaigns.groupby("Evidence", sort=False):
    worst = table.groupby(["Run", "kind"])[
        ["detection_rate", "targeted_check_detection_rate", "vector_match_rate"]
    ].min()
    fig, axes = plt.subplots(3, 1, figsize=(13, 3.2 + 1.1 * table["Run"].nunique()),
                             sharex=True, layout="constrained")
    for ax, (metric_name, label) in zip(axes, (
        ("detection_rate", "Score decreased"),
        ("targeted_check_detection_rate", "Intended check detected"),
        ("vector_match_rate", "Exact check vector matched"),
    )):
        sns.heatmap(worst[metric_name].unstack("kind").rename(columns=title),
                    cmap="YlGn", vmin=0, vmax=1, annot=True, fmt=".0%",
                    ax=ax, cbar=False, yticklabels=True)
        ax.set(title=label, xlabel="", ylabel="", facecolor="#eeeeee")
        ax.tick_params(axis="y", labelrotation=0, labelsize=9)
    axes[-1].tick_params(axis="x", rotation=16)
    fig.suptitle(f"{family} · lowest measured agreement across tested rates")
    plt.show()

full_rate = campaigns.loc[campaigns["requested_rate"] == max(RATES)].copy()
effective = full_rate.pivot(index="Run", columns="kind", values="effective_dataset_rate")
fig, ax = plt.subplots(figsize=(14, 6.3), layout="constrained")
sns.heatmap(effective.rename(columns=title), cmap="PuBu", vmin=0, vmax=1, annot=True, fmt=".0%",
            cbar_kws={"format": PercentFormatter(1)}, ax=ax, yticklabels=True)
ax.set(title=f"Share of all tasks actually corrupted at {max(RATES):.0%} requested",
       xlabel="", ylabel="", facecolor="#eeeeee")
ax.tick_params(axis="x", rotation=23)
plt.show()
columns = ["detection_rate", "targeted_check_detection_rate", "vector_match_rate"]
max_rate_table = full_rate[["Run", "kind", "corrupted_tasks", "eligible_tasks", "effective_dataset_rate",
                            *columns, "clean_tasks", "false_positive_rate", "score_error", "passed"]]
detail_table = max_rate_table.rename(columns={
    "kind": "Campaign", "detection_rate": "Score decreased",
    "targeted_check_detection_rate": "Target detected", "vector_match_rate": "Exact vector",
    "effective_dataset_rate": "Share of dataset corrupted", "false_positive_rate": "Clean false positives",
}).style.hide(axis="index").format({
    "Score decreased": pct, "Target detected": pct, "Exact vector": pct,
    "Share of dataset corrupted": pct, "Clean false positives": pct, "score_error": "{:+.2e}",
})
display(HTML(f"<details><summary>Campaign counts and score errors at {max(RATES):.0%} requested corruption</summary>"
             f"{detail_table.to_html()}</details>"))

# Rows at every rate are available in the generated campaigns.csv.
for family in campaigns["Evidence"].unique():
    subset = campaigns.loc[(campaigns["Evidence"] == family) & campaigns["false_positive_rate"].notna()]
    if not subset.empty:
        ax = sns.lineplot(data=subset, x="requested_rate", y="false_positive_rate", hue="Run",
                          marker="o", estimator=None, units="kind", legend="full")
        ax.set(title=f"{family} · clean false positives by rate", xlabel="Requested corruption rate",
               ylabel="False-positive rate", ylim=(-.04, 1.04))
        ax.xaxis.set_major_formatter(PercentFormatter(1))
        ax.yaxis.set_major_formatter(PercentFormatter(1))
        plt.show()

## 5. Investigate failures, exclusions, and coverage gaps

A campaign exclusion means a particular mutation could not be exercised on that task. A baseline finding gives a concrete task/check to inspect. For example, the November 4 report found retail lookups returning “User not found” and flagged the need to review task definitions, database snapshots, and evaluator behavior before assigning a cause. Current findings below come from this run. The full JSON retains per-task checks, replay evidence, and action details.

In [ ]:
exclusions = pd.DataFrame(
    [{"Run": name, **item} for name, (_, report, _) in runs.items()
     for item in report["excluded_mutations"]]
)
unexercised = [
    {"Run": name, "Campaign": title(kind)}
    for name, (_, report, _) in runs.items() for kind in report["unexercised_campaigns"]
]
if unexercised:
    display(Markdown("**Unexercised campaigns**"))
    display(pd.DataFrame(unexercised))
if not exclusions.empty:
    display(Markdown("**Why mutations were excluded**"))
    display(exclusions.groupby(["Run", "kind", "reason"]).size().rename("Tasks").reset_index())
    exclusions.to_csv(RUN_DIR / "exclusions.csv", index=False)

evidence = [{"Run": name, **item} for name, (_, report, _) in runs.items()
            for item in findings(report, limit=3)]
if evidence:
    display(Markdown("**Task findings · up to three per run**"))
    evidence_table = pd.DataFrame(evidence).astype(object)
    display(evidence_table.where(evidence_table.notna(), "—"))

gaps = [{"Run": name, "Uncovered outcome requirement": item}
        for name, (_, report, _) in runs.items()
        for item in report["baseline_summary"]["outcome_uncovered"]]
if gaps:
    display(Markdown("**Uncovered outcome requirements**"))
    display(pd.DataFrame(gaps))

display(Markdown("**Inputs and report artifacts**"))
provenance = []
for name, (_, report, directory) in runs.items():
    manifest = manifests[name]
    if manifest.get("task_drift"):
        print(f"{name}: preserved historical task definitions for {', '.join(manifest['task_drift'])}.")
    provenance.append({
        "Run": name, "Evaluator": report["evaluator"], "Dataset": manifest.get("dataset", "Controlled demo"),
        "Split": manifest.get("split", "—"), "Manifest": str(directory.relative_to(ROOT) / "manifest.json"),
        "Report": str(directory.relative_to(ROOT) / "report.md"),
    })
display(pd.DataFrame(provenance))

## 6. Validate tool-usage diversity

Diversity is Shannon entropy divided by the log of the number of tools in a fixed reference inventory, `H / ln(K)`. A score near 1 means calls are spread evenly across that inventory. All recognized calls to one tool give a score of 0. Unused tools remain in the inventory, unknown calls are counted separately, and a run with zero recognized calls has no score.

Most inputs are **reference calls**. Synthetic calls come from the matched fixture's expected actions and τ² calls come from the original reference actions, assistant only. Synthetic inventories come from YAML plus built-in system tools. τ² inventories come from the assistant tool registry. The recorded sample uses its actual assistant actions. Scores describe balance within each domain's inventory. The companion `tool_usage_diversity.ipynb` also supports recorded assistant traces.

Two perturbations answer different questions. Keeping calls to one global tool should force a zero score. Keeping tasks that individually use one tool can still yield diversity if those tasks use different tools. The latter subset can also be empty.

In [ ]:
from loguru import logger

logger.disable("tau2")


def calls_for_diversity(manifest, family):
    entries = manifest["tasks"]
    first = entries[0]["task"]
    if family == "Synthetic static":
        inventory = sorted(first["tools"].keys() | SYSTEM.keys())
        actions = [(entry["task"].get("python_fixture") or {}).get("attributes", {}).get("expected_tool_calls")
                   for entry in entries]
    else:
        _, validators, _ = context(json.dumps(first["source"], sort_keys=True))
        inventory = sorted(validators["assistant"])
        if family == "τ² recorded":
            actions = [entry["task"]["candidate_actions"] for entry in entries]
        else:
            actions = [(entry["task"]["definition"].get("evaluation_criteria") or {}).get("actions")
                       for entry in entries]
    task_calls = [
        {"task_id": entry["id"], "calls": [
            call["name"] for call in (calls if isinstance(calls, list) else [])
            if isinstance(call, dict) and isinstance(call.get("name"), str)
            and (call.get("requestor") or "assistant") == "assistant"
        ]}
        for entry, calls in zip(entries, actions)
    ]
    return inventory, task_calls


metric = "normalized_tool_usage_diversity"
assert close(calculate_diversity({"a": 1, "b": 1}, ["a", "b"])[metric], 1)
assert close(calculate_diversity({"a": 5}, ["a", "b"])[metric], 0)
assert calculate_diversity({}, ["a", "b"])[metric] is None

diversity_rows, diversity_details = [], {}
for name, (family, _, _) in runs.items():
    if family not in ("Synthetic static", "τ² original", "τ² recorded"):
        continue
    inventory, task_calls = calls_for_diversity(manifests[name], family)
    counts = Counter(call for task in task_calls for call in task["calls"])
    metrics = calculate_diversity(counts, inventory)
    perturbation = validate_perturbations({**metrics, "task_calls": task_calls}, inventory)
    global_subset = perturbation["strict_single_tool"]
    single_tasks = perturbation["task_level_single_tool_subset"]
    if metrics["recognized_tool_calls"]:
        assert global_subset["passed"], name
    diversity_details[name] = {"metrics": metrics, "perturbations": perturbation, "task_calls": task_calls}
    diversity_rows.append({
        "Run": name, "Inventory": len(inventory), "Used tools": metrics["distinct_tools_used"],
        "Recognized calls": metrics["recognized_tool_calls"], "Unknown calls": metrics["unknown_tool_calls"],
        "Tasks with calls": sum(bool(task["calls"]) for task in task_calls),
        "One-tool tasks": single_tasks["task_count"], "Baseline": metrics[metric],
        "One global tool": global_subset[metric], "One-tool task subset": single_tasks[metric],
    })

(RUN_DIR / "tool-diversity.json").write_text(json.dumps(diversity_details, indent=2, allow_nan=False))
diversity = pd.DataFrame(diversity_rows)
variants = ["Baseline", "One global tool", "One-tool task subset"]
display(diversity.style.hide(axis="index").format({name: "{:.3f}" for name in variants}, na_rep="—"))

long = diversity.melt(id_vars="Run", value_vars=variants, var_name="Sample", value_name="Diversity")
fig, ax = plt.subplots(figsize=(10, 5.5), layout="constrained")
sns.barplot(data=long, y="Run", x="Diversity", hue="Sample", ax=ax)
ax.set(xlim=(0, 1), title="Tool usage diversity under concentration", ylabel="")
ax.legend(loc="lower right")
plt.show()

columns = 4
rows = (len(diversity_details) + columns - 1) // columns
fig, axes = plt.subplots(rows, columns, squeeze=False, figsize=(16, 2.9 * rows), layout="constrained")
for ax, (name, evidence) in zip(axes.flat, diversity_details.items()):
    top = Counter(evidence["metrics"]["tool_call_counts"]).most_common(5)
    tools = [(tool, count) for tool, count in top if count]
    if tools:
        ax.barh([tool for tool, _ in tools], [count for _, count in tools], color="#0072b2")
        ax.invert_yaxis()
    ax.tick_params(axis="y", labelsize=7)
    ax.set_title(name.replace("_", " "), fontsize=9)
    ax.set_xlabel("Calls", fontsize=8)
for ax in list(axes.flat)[len(diversity_details):]:
    ax.set_visible(False)
fig.suptitle("Five most-used recognized tools per domain")
plt.show()

## 7. Validate semantic correctness with an LLM judge

The judge sees the task, expected outcomes, relevant workflow and policy evidence, and tool definitions. It rates task consistency, expected outcome validity, policy compliance, and workflow feasibility. Each dimension receives 1 for supported correctness, 0.5 for uncertainty, or 0 for a supported defect, with an explanation, source references, and verbatim quotes when available. The task score is the mean of the four dimensions. Each domain score is the mean across its selected tasks.

Three constructed clean tasks anchor four corruption campaigns. Each campaign changes one field and declares an expected zero for its target dimension. The harness compares the observed score and full check vector with that expectation, tracks false positives on clean controls, and reports the number of eligible tasks. The τ² inputs include the task definition, policy, and tool schemas. Database state outside the supplied task definition is unavailable to the judge. The sampled reference tasks have no independent semantic labels. Their scores describe the judge's assessments. The operational scores shown alongside them use the same task IDs.

Set `RUN_JUDGE = True` to regenerate the results. This uses the project API configuration and sends evidence to the configured endpoint. The default sample size is one task per domain. A value of zero scores all tasks.

In [9]:
from validation.correctness_checker import LLM_CHECKS

RUN_JUDGE = False
JUDGE_LIMIT = 1
JUDGE_REPORT = ROOT / "output" / "llm-judge-notebook" / "judge_report.json"

if RUN_JUDGE:
    from evaluation.llm_judge import Judge
    from validation.llm_judge_validation import run as run_judge

    JUDGE_REPORT = RUN_DIR / "llm-judge" / "judge_report.json"
    judge_report = run_judge(
        ROOT, TAU_ROOT, Judge.from_env(ROOT / ".env"),
        JUDGE_REPORT.parent, limit=JUDGE_LIMIT, seed=SEED,
    )
elif JUDGE_REPORT.is_file():
    judge_report = json.loads(JUDGE_REPORT.read_text())
else:
    judge_report = None
    print("Judge report unavailable. Set RUN_JUDGE = True to generate a new report.")


In [10]:
if judge_report is not None:
    controls = judge_report["controls"]
    judge_info = judge_report["judge"]
    print(f"{judge_info['provider']} · {judge_info['requested_model']} · {judge_info['prompt_version']}")
    print(f"Control validation {controls['status'].upper()} · "
          f"clean baseline {pct(controls['baseline_score'])} · "
          f"clean false positives {pct(controls['baseline_false_positive_rate'])} "
          f"({controls['clean_labeled_tasks']} controls)")
    campaign_table = pd.DataFrame([{
        "Campaign": title(row["kind"]),
        "Requested rate": pct(row["requested_rate"]),
        "Corrupted / eligible": f"{row['corrupted_tasks']}/{row['eligible_tasks']}",
        "Target detected": f"{row['targeted_detected']}/{row['corrupted_tasks']}",
        "Exact vectors": f"{row['expected_check_vectors_matched']}/{row['corrupted_tasks']}",
        "Expected mean": pct(row["expected_score"]),
        "Observed mean": pct(row["observed_score"]),
    } for row in controls["campaigns"]])
    display(campaign_table)

    sample_rows, dimension_rows, judgment_rows = [], [], []
    family_scores = {}
    for dataset in judge_report["datasets"]:
        name = dataset["dataset"]
        judgments = dataset["judgments"]
        family = "τ²-bench" if name.startswith("tau2/") else "Synthetic"
        family_scores.setdefault(family, []).extend(row["score"] for row in judgments.values())
        run_name = name.replace("tau2/", "τ² ")
        operational = runs.get(run_name, (None, {}, None))[1].get("baseline_results", {})
        matched_scores = [operational[key]["score"] for key in judgments if key in operational]
        sample_rows.append({
            "Dataset": name,
            "Judged / available": f"{len(judgments)}/{dataset['population']}",
            "Judge mean": pct(dataset["mean_score"]),
            "Operational mean on judged tasks": (
                pct(sum(matched_scores) / len(matched_scores))
                if len(matched_scores) == len(judgments) else "—"
            ),
        })
        for dimension, counts in dataset["score_distribution"].items():
            dimension_rows.append({
                "Dataset": name, "Dimension": title(dimension),
                "Correct": counts.get("1.0", 0), "Uncertain": counts.get("0.5", 0),
                "Incorrect": counts.get("0.0", 0),
            })
        for task_id, result in judgments.items():
            judgment_rows.append({"Dataset": name, "Task ID": task_id,
                                  **{title(key): value for key, value in result["checks"].items()}})
    if sample_rows:
        display(pd.DataFrame([
            {"Dataset": family, "Judged tasks": len(scores), "Sample mean": pct(sum(scores) / len(scores))}
            for family, scores in family_scores.items()
        ]))
        display(pd.DataFrame(sample_rows))
        distribution_table = pd.DataFrame(dimension_rows)
        display(distribution_table.groupby("Dimension", sort=False)[
            ["Correct", "Uncertain", "Incorrect"]
        ].sum())
        with pd.option_context("display.max_rows", 20):
            display(pd.DataFrame(judgment_rows))

    examples = [item for item in controls["audit"]
                if item["corrupted"] and item["rate"] == 1]
    example_rows = []
    for kind in LLM_CHECKS:
        candidates = [item for item in examples if item["kind"] == kind]
        if candidates:
            example = next((item for item in candidates if item["result"]["checks"] != {
                **item["baseline"]["checks"], kind: 0,
            }), candidates[0])
            dimension = example["result"]["dimensions"][kind]
            example_rows.append({
                "Control": title(kind), "Score": dimension["score"],
                "Explanation": dimension["explanation"],
                "Evidence": " | ".join(
                    f"{e['field']}: {e['quote']}" if e["quote"] else e["field"]
                    for e in dimension["evidence"]
                ),
            })
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(example_rows))
    print("Per-task explanations, source references, and input fingerprints are in judge_report.json.")


openai · gpt-6-luna · semantic-correctness-v3
Control validation FAIL · clean baseline 100.0% · clean false positives 0.0% (3 controls)
Per-task explanations, source references, and input fingerprints are in judge_report.json.


,Campaign,Requested rate,Corrupted / eligible,Target detected,Exact vectors,Expected mean,Observed mean
0,expected outcome,0.0%,0/3,0/0,0/0,100.0%,100.0%
1,expected outcome,33.3%,1/3,1/1,1/1,91.7%,91.7%
2,expected outcome,66.7%,2/3,2/2,1/2,83.3%,75.0%
3,expected outcome,100.0%,3/3,3/3,1/3,75.0%,58.3%
4,policy compliance,0.0%,0/3,0/0,0/0,100.0%,100.0%
5,policy compliance,33.3%,1/3,1/1,0/1,91.7%,87.5%
6,policy compliance,66.7%,2/3,2/2,1/2,83.3%,79.2%
7,policy compliance,100.0%,3/3,3/3,1/3,75.0%,66.7%
8,task consistency,0.0%,0/3,0/0,0/0,100.0%,100.0%
9,task consistency,33.3%,1/3,1/1,0/1,91.7%,75.0%


,Dataset,Judged tasks,Sample mean
0,Synthetic,7,21.4%
1,τ²-bench,3,50.0%


,Dataset,Judged / available,Judge mean,Operational mean on judged tasks
0,banking_accounts_deposits,1/8,12.5%,100.0%
1,banking_customer_onboarding,1/6,25.0%,75.0%
2,banking_fraud_monitoring,1/13,25.0%,100.0%
3,healthcare_patient_onboarding,1/9,37.5%,100.0%
4,healthcare_prior_authorization,1/9,12.5%,75.0%
5,insurance_claims_intake,1/10,25.0%,100.0%
6,manufacturing_purchase_to_pay,1/9,12.5%,100.0%
7,tau2/retail,1/114,25.0%,100.0%
8,tau2/airline,1/50,50.0%,100.0%
9,tau2/telecom,1/114,75.0%,100.0%


,Correct,Uncertain,Incorrect
Dimension,,,
task consistency,3,2,5
expected outcome,0,2,8
policy compliance,2,7,1
workflow feasibility,0,3,7


,Dataset,Task ID,task consistency,expected outcome,policy compliance,workflow feasibility
0,banking_accounts_deposits,DigitalAccountOpening_W1_TC2,0.0,0.0,0.5,0.0
1,banking_customer_onboarding,BankOnboard_W7_TC1,0.5,0.0,0.5,0.0
2,banking_fraud_monitoring,BankingFraud_W7_TC1,0.5,0.0,0.5,0.0
3,healthcare_patient_onboarding,HealthcareOnboarding_W1_TC2,1.0,0.0,0.5,0.0
4,healthcare_prior_authorization,HealthcarePA_W1_TC2,0.0,0.0,0.5,0.0
5,insurance_claims_intake,ClaimsAgent_W1_TC2,0.0,0.0,1.0,0.0
6,manufacturing_purchase_to_pay,P2P_W1_TC2,0.0,0.0,0.5,0.0
7,tau2/retail,81,0.0,0.0,0.5,0.5
8,tau2/airline,40,1.0,0.5,0.0,0.5
9,tau2/telecom,[mms_issue]bad_network_preference|break_app_sm...,1.0,0.5,1.0,0.5


,Control,Score,Explanation,Evidence
0,task consistency,0,"The milestone requires approval before recording, while the request and reference workflow explicitly prohibit approval.",milestone: The item must be approved before it is recorded. | request: Do not approve it. | reference_workflow: Do not approve the item.
1,expected outcome,0,"The declared outcome contradicts the objective: recording for review leaves the item pending review, not approved or final.",request: Record an invoice for review. Do not approve it. | tools: records the item as pending review without approving it. | expected_outcome: approved and marked final.
2,policy compliance,0,"The reference workflow calls for recording the item despite the stated policy requiring explicit submitter consent first, and the conversation says consent has not been asked for.",policy: Get explicit submitter consent before calling record_for_review. | conversation: The submitter has not been asked for consent. | reference_workflow: Call record_for_review once. Do not approve the item.
3,workflow feasibility,0,"The reference workflow cannot run as specified: the tool only records archived items, while this item is active and no tool can archive it.","tools: record_for_review can only record archived items. This item is active, and no tool can archive it. | reference_workflow: Call record_for_review once. Do not approve the item."


## 8. What the results establish

A correctness validation pass applies to the tested corruption rules, seed, and task set. The original benchmark task scores still reflect any defects or unavailable checks in the baseline. The control runs estimate false positives on constructed cases. Human-reviewed original-task labels are needed to estimate false positives on original tasks. Retail and airline outcome coverage depends on available independent assertions. Synthetic static findings need review against the fixture sources before drawing conclusions about real replay.

### Research context

- The October advisor discussion emphasized inspecting trajectories, validating deterministic checks against known cases, and exploring a judge or jury for questions that require contextual reasoning.
- The November 4 snapshot reported 15 retail tasks with operational defects, 0/60 baseline false positives in the three constructed control sets, and an inconclusive static banking-onboarding run with a 62.5% mean score. Those figures describe the archived snapshot. Use the fresh tables above for this run.
- The semantic control experiment measures whether a single judge recognizes the four constructed defect types and how often it flags clean controls. The reference-task sample shows the mix of uncertainty and identified defects for the sampled inputs.

The next research questions are whether task-level adjudication explains the baseline defects, whether more independent outcome assertions expand coverage, and whether the same results hold across broader mutation variants and reviewed labels.

In [ ]:
for family, group in overview.groupby("Evidence", sort=False):
    statuses = group["Validation"].value_counts()
    summary = ", ".join(f"{count} {status.lower()}" for status, count in statuses.items())
    run_label = "run" if len(group) == 1 else "runs"
    print(f"{family}: {len(group)} {run_label}, {group['Tasks'].sum()} tasks · {summary}")
for family in ("Synthetic static", "τ² original"):
    group = overview.loc[overview["Evidence"] == family]
    if group.empty:
        continue
    lowest = group["Mean score"].idxmin()
    covered = (group["Full coverage"] * group["Tasks"]).round().sum()
    display(Markdown(
        f"**{family}.** The lowest baseline mean was **{group.loc[lowest, 'Mean score']:.1%}** "
        f"for {lowest.replace('_', ' ')}. The evaluator marked **{int(covered)}/{group['Tasks'].sum()}** "
        "tasks fully evaluated."
    ))
control_rows = overview.loc[overview["Evidence"] == "τ² controls"]
if not control_rows.empty:
    clean_count = int(control_rows["Clean labels"].sum())
    flagged = int((control_rows["Clean false positives"] * control_rows["Clean labels"]).sum())
    display(Markdown(f"**Constructed controls.** {flagged}/{clean_count} labeled clean tasks "
                     "were flagged at baseline across the three τ² domains."))
print(f"Full reports, audit evidence, CSVs, and input fingerprints: {RUN_DIR}")